In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import math
import os

import numpy as np
import torch
from torch_geometric import seed_everything

seed_everything(42)

cache_dir = "./cash_predql"

/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing
/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_cluster/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing
/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while import

In [3]:
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_device()
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from relbench.datasets import get_dataset
from relbench.modeling.utils import get_stype_proposal
from predql_tasks import StatsUserEngagementTmpTask

In [5]:
dataset = get_dataset("ctu-stats", download=False)
task = StatsUserEngagementTmpTask()
db = dataset.get_db()

Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 1.06 seconds.


In [6]:
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

{'postHistory': {'__PK__': <stype.numerical: 'numerical'>,
  'PostHistoryTypeId': <stype.numerical: 'numerical'>,
  'PostId': <stype.numerical: 'numerical'>,
  'RevisionGUID': <stype.text_embedded: 'text_embedded'>,
  'CreationDate': <stype.timestamp: 'timestamp'>,
  'UserId': <stype.numerical: 'numerical'>,
  'Text': <stype.text_embedded: 'text_embedded'>,
  'Comment': <stype.text_embedded: 'text_embedded'>,
  'UserDisplayName': <stype.text_embedded: 'text_embedded'>,
  'FK_posts_PostId': <stype.numerical: 'numerical'>,
  'FK_users_UserId': <stype.numerical: 'numerical'>},
 'votes': {'__PK__': <stype.numerical: 'numerical'>,
  'PostId': <stype.numerical: 'numerical'>,
  'VoteTypeId': <stype.numerical: 'numerical'>,
  'CreationDate': <stype.timestamp: 'timestamp'>,
  'UserId': <stype.numerical: 'numerical'>,
  'BountyAmount': <stype.categorical: 'categorical'>,
  'FK_posts_PostId': <stype.numerical: 'numerical'>,
  'FK_users_UserId': <stype.numerical: 'numerical'>},
 'tags': {'__PK__':

In [7]:
from sentence_transformers import SentenceTransformer

class GloveTextEmbedding:
    def __init__(self, device: torch.device=None):
        self.model = SentenceTransformer(
            "sentence-transformers/average_word_embeddings_glove.6B.300d",
            device=device,
        )

    def __call__(self, sentences: list[str]) -> torch.Tensor:
        filt_sentences = [str(s) if s is not None and str(s) != 'nan' else "" for s in sentences]
        return torch.from_numpy(self.model.encode(filt_sentences))

In [8]:
from getpass import getpass

import relbench.modeling.graph
import relbench.modeling.utils
from relbench.modeling.graph import make_pkey_fkey_graph
from torch_frame.config.text_embedder import TextEmbedderConfig

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Insert HF token: ")

text_embedder = TextEmbedderConfig(
    text_embedder=GloveTextEmbedding(device=device), batch_size=256
)

def patched_to_unix_time(ser):
    unix_time = ser.astype("int64").values.copy()
    if ser.dtype == np.dtype("datetime64[ns]"):
        unix_time //= 10**9
    return unix_time

relbench.modeling.graph.to_unix_time = patched_to_unix_time
relbench.modeling.utils.to_unix_time = patched_to_unix_time


data, col_stats_dict = make_pkey_fkey_graph(
    db, 
    col_to_stype_dict, 
    text_embedder,
    cache_dir=os.path.join(cache_dir, "stat_user_engagement_tmp_cache")
)


/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-packages/relbench/modeling/graph.py:93: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  pkey_index = torch.from_numpy(pkey_index[mask].astype(int).values)
/home/kolesiko/CTU/BT/PredQL/.venv/lib/python3.12/site-

In [9]:
data

HeteroData(
  postHistory={
    tf=TensorFrame([303187, 8]),
    time=[303187],
  },
  votes={
    tf=TensorFrame([328064, 5]),
    time=[328064],
  },
  tags={ tf=TensorFrame([1032, 4]) },
  posts={
    tf=TensorFrame([91976, 20]),
    time=[91976],
  },
  badges={
    tf=TensorFrame([79851, 3]),
    time=[79851],
  },
  users={
    tf=TensorFrame([40325, 13]),
    time=[40325],
  },
  postLinks={
    tf=TensorFrame([11102, 4]),
    time=[11102],
  },
  comments={
    tf=TensorFrame([174305, 6]),
    time=[174305],
  },
  (postHistory, f2p_FK_posts_PostId, posts)={ edge_index=[2, 303187] },
  (posts, rev_f2p_FK_posts_PostId, postHistory)={ edge_index=[2, 303187] },
  (postHistory, f2p_FK_users_UserId, users)={ edge_index=[2, 281859] },
  (users, rev_f2p_FK_users_UserId, postHistory)={ edge_index=[2, 281859] },
  (votes, f2p_FK_posts_PostId, posts)={ edge_index=[2, 328064] },
  (posts, rev_f2p_FK_posts_PostId, votes)={ edge_index=[2, 328064] },
  (votes, f2p_FK_users_UserId, users)={ e

In [10]:
def get_transform(table, task):
    target_map = torch.from_numpy(table.df["label"].values)

    if task.task_type in ["binary_classification", "multiclass_classification"]:
        target_map = target_map.to(torch.long)
    else:
        target_map = target_map.to(torch.float)
    
    def transform(batch):
        batch.y = target_map[batch[task.entity_table].input_id]
        return batch

    return transform

In [11]:
from torch_geometric.loader import NeighborLoader

loader_dict = {}

for split in ["train", "val", "test"]:
    table = task.get_table(split)

    times = table.df[task.time_col].values.astype('datetime64[s]').astype('int64')
    
    loader_dict[split] = NeighborLoader(
        data,
        num_neighbors=[128, 128],
        input_nodes=(
            task.entity_table,
            torch.from_numpy(table.df[task.entity_col].values).to(torch.long)
        ),
        input_time=torch.from_numpy(times).to(torch.long),
        time_attr="time",
        transform=get_transform(table, task),
        batch_size=512,
        temporal_strategy="uniform",
        shuffle=split == "train",
        num_workers=0,
        persistent_workers=False
    )

Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 1.09 seconds.
Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 0.98 seconds.


In [13]:
from typing import Any
import torch
from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE, HeteroTemporalEncoder
from torch import Tensor
from torch.nn import Embedding, ModuleDict
from torch_frame.data.stats import StatType
from torch_geometric.data import HeteroData
from torch_geometric.nn import MLP
from torch_geometric.typing import NodeType


class Model(torch.nn.Module):

    def __init__(
        self,
        data: HeteroData,
        col_stats_dict: dict[str, dict[str, dict[StatType, Any]]],
        num_layers: int,
        channels: int,
        out_channels: int,
        aggr: str,
        norm: str,
        # List of node types to add shallow embeddings to input
        shallow_list: list[NodeType] = [],
        # ID awareness
        id_awareness: bool = False,
    ):
        super().__init__()

        self.encoder = HeteroEncoder(
            channels=channels,
            node_to_col_names_dict={
                node_type: data[node_type].tf.col_names_dict
                for node_type in data.node_types
            },
            node_to_col_stats=col_stats_dict,
        )
        self.temporal_encoder = HeteroTemporalEncoder(
            node_types=[
                node_type for node_type in data.node_types if "time" in data[node_type]
            ],
            channels=channels,
        )
        self.gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        )
        self.head = MLP(
            channels,
            out_channels=out_channels,
            norm=norm,
            num_layers=1,
        )
        self.embedding_dict = ModuleDict(
            {
                node: Embedding(data.num_nodes_dict[node], channels)
                for node in shallow_list
            }
        )

        self.id_awareness_emb = None
        if id_awareness:
            self.id_awareness_emb = torch.nn.Embedding(1, channels)
        self.reset_parameters()

    def reset_parameters(self):
        self.encoder.reset_parameters()
        self.temporal_encoder.reset_parameters()
        self.gnn.reset_parameters()
        self.head.reset_parameters()
        for embedding in self.embedding_dict.values():
            torch.nn.init.normal_(embedding.weight, std=0.1)
        if self.id_awareness_emb is not None:
            self.id_awareness_emb.reset_parameters()

    def forward(
        self,
        batch: HeteroData,
        entity_table: NodeType,
    ) -> Tensor:
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )

        return self.head(x_dict[entity_table][: seed_time.size(0)])

    def forward_dst_readout(
        self,
        batch: HeteroData,
        entity_table: NodeType,
        dst_table: NodeType,
    ) -> Tensor:
        if self.id_awareness_emb is None:
            raise RuntimeError(
                "id_awareness must be set True to use forward_dst_readout"
            )
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)
        # Add ID-awareness to the root node
        x_dict[entity_table][: seed_time.size(0)] += self.id_awareness_emb.weight

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
        )

        return self.head(x_dict[dst_table])


model = Model(
    data=data,
    col_stats_dict=col_stats_dict,
    num_layers=2,
    channels=128,
    out_channels=1,
    aggr="sum",
    norm="batch_norm",
).to(device)

# pos_weight = torch.tensor([10.0]).to(device)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
epochs = 20

In [14]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix

from tqdm import tqdm

def train_epoch():
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader_dict["train"], desc="Training"):
        batch = batch.to(device)
        optimizer.zero_grad()
        
        logits = model(batch, task.entity_table).squeeze(-1)
        
        y = batch.y.to(torch.float)
        logits = logits[:y.size(0)] 
        
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(loader_dict["train"])

@torch.no_grad()
def evaluate(split: str):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in tqdm(loader_dict[split], desc=f"Evaluating {split}"):
        batch = batch.to(device)
        logits = model(batch, task.entity_table).squeeze(-1)
        
        y = batch.y.to(torch.float)
        logits = logits[:y.size(0)]
        
        probs = torch.sigmoid(logits)
        
        all_preds.append(probs.cpu())
        all_labels.append(y.cpu())
        
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    
    binary_preds = (preds > 0.5).astype(int)
    
    return {
        "auroc": roc_auc_score(labels, preds),
        "ap": average_precision_score(labels, preds),
        "f1": f1_score(labels, binary_preds),
        "confusion": confusion_matrix(labels, binary_preds).tolist()
    }

In [15]:
for epoch in range(1, 21):
    loss = train_epoch()
    val_metrics = evaluate("val")
    
    print(f"Epoch {epoch:02d} | Loss: {loss:.4f} | Val AUROC: {val_metrics['auroc']:.4f} | Val F1: {val_metrics['f1']:.4f}")

test_metrics = evaluate("test")
print(f"Test AUROC: {test_metrics['auroc']:.4f}\nTest AP: {test_metrics['ap']:.4f}\nTest F1: {test_metrics['f1']:.4f}\nConfusion Matrix: {test_metrics['confusion']}")

Training:   0%|          | 0/123 [00:00<?, ?it/s]

Evaluating val: 100%|██████████| 26/26 [00:00<00:00, 38.35it/s]


Epoch 01 | Loss: 0.3457 | Val AUROC: 0.8702 | Val F1: 0.4085


Evaluating val: 100%|██████████| 26/26 [00:00<00:00, 43.45it/s]


Epoch 02 | Loss: 0.2922 | Val AUROC: 0.8661 | Val F1: 0.4686


...

Epoch 03 | Loss: 0.2852 | Val AUROC: 0.8625 | Val F1: 0.4485


Evaluating val: 100%|██████████| 26/26 [00:00<00:00, 39.55it/s]


Epoch 04 | Loss: 0.2807 | Val AUROC: 0.8612 | Val F1: 0.4555


Training:  62%|██████▏   | 76/123 [00:04<00:02, 17.08it/s]


KeyboardInterrupt: 

In [16]:
test_metrics = evaluate("test")
print(f"Test AUROC: {test_metrics['auroc']:.4f}\nTest AP: {test_metrics['ap']:.4f}\nTest F1: {test_metrics['f1']:.4f}\nConfusion Matrix: {test_metrics['confusion']}")

Evaluating test: 100%|██████████| 29/29 [00:00<00:00, 38.91it/s]

Test AUROC: 0.8613
Test AP: 0.4637
Test F1: 0.4250
Confusion Matrix: [[12791, 430], [1007, 531]]
